# 16 Final checks (run once)

The hold-out was kept unread until this notebook. Models and settings were fixed before looking:

- **Class 2 vs 3:** logistic regression on the 15 shape features (no tuned model beat it in 15). 160 dev segments to fit, 40 hold-out segments to score. **Not clean:** these 40 segments were part of the earlier chunk-level 2 vs 3 development (notebooks 04 to 07), so the hold-out is not fully fresh for this question.
- **Seizure vs classes 2 and 3:** logistic regression on all 23 features. 240 dev segments to fit, 60 hold-out segments (20 seizures) to score.

Each is run once and reported as it comes out. Nothing is changed afterwards. Intervals: 1,000 bootstrap draws that resample whole twin groups.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import PROCESSED, RANDOM_STATE
from src.features import FEATURES, SHAPE_FEATURES

dev = pd.read_csv(PROCESSED / "features_dev.csv")
hold = pd.read_csv(PROCESSED / "features_holdout.csv")         # first and only read of the hold-out features
assert set(dev["segment"]).isdisjoint(hold["segment"]) and set(dev["group"]).isdisjoint(hold["group"])

def bootstrap_auc(y, p, group, n_draws=1000, seed=RANDOM_STATE):
    """AUC over draws that resample whole twin groups with replacement. Draws with one class only are skipped."""
    rng = np.random.default_rng(seed)
    idx_of = {g: np.flatnonzero(group == g) for g in np.unique(group)}
    keys = list(idx_of)
    out = []
    for _ in range(n_draws):
        pick = rng.choice(len(keys), size=len(keys), replace=True)
        idx = np.concatenate([idx_of[keys[i]] for i in pick])
        if len(np.unique(y[idx])) == 2:
            out.append(roc_auc_score(y[idx], p[idx]))
    return np.array(out)

def final_check(name, classes, positive, cols):
    d, h = dev[dev["y"].isin(classes)], hold[hold["y"].isin(classes)]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(d[cols], d["y"] == positive)
    p = model.predict_proba(h[cols])[:, 1]
    y = (h["y"] == positive).to_numpy()
    pred = p > 0.5
    boot = bootstrap_auc(y, p, h["group"].to_numpy())
    return {"check": name, "hold-out segments": len(h), "positives": int(y.sum()), "hold-out groups": h["group"].nunique(),
            "AUC": roc_auc_score(y, p), "AUC 95% interval": f"{np.percentile(boot, 2.5):.3f} to {np.percentile(boot, 97.5):.3f}",
            "valid draws": len(boot), "sensitivity @0.5": pred[y].mean(), "specificity @0.5": (~pred[~y]).mean()}

final = pd.DataFrame([
    final_check("class 2 vs 3, shape features (NOT CLEAN)", (2, 3), 2, SHAPE_FEATURES),
    final_check("seizure vs classes 2 and 3, all 23 features", (1, 2, 3), 1, FEATURES),
]).set_index("check")
final.round(3).T

check,"class 2 vs 3, shape features (NOT CLEAN)","seizure vs classes 2 and 3, all 23 features"
hold-out segments,40,60
positives,20,20
hold-out groups,18,32
AUC,0.883,0.988
AUC 95% interval,0.771 to 1.000,0.952 to 1.000
valid draws,1000,1000
sensitivity @0.5,0.7,0.9
specificity @0.5,0.95,0.925


## Findings (reported as they came out)

| | class 2 vs 3 (NOT CLEAN) | seizure vs classes 2 and 3 |
|---|---|---|
| Hold-out segments (groups) | 40 (18) | 60 (32) |
| AUC | 0.883 | 0.988 |
| 95% interval (1,000 group draws) | 0.771 to 1.000 | 0.952 to 1.000 |
| Sensitivity at 0.5 | 0.70 | 0.90 |
| Specificity at 0.5 | 0.95 | 0.925 |

- **Seizure vs depth classes** matches development (0.997 there, 0.988 here). Sensitivity of 0.90 means 2 of 20 seizures missed.
- **Class 2 vs 3** came out higher than in development (0.883 against about 0.75). I do not read this as the model being better than I thought. Only 40 segments in 18 groups, and the interval is wide (0.77 to 1.00), which includes the development score. A lucky draw is the most likely explanation. The honest summary is "above chance, size uncertain".
- This check is **not clean**: these 40 segments were part of the earlier chunk-level 2 vs 3 development.

## Caveats

- Small hold-outs. 20 positives each.
- Hold-out and development come from the same pool, with the same patient-overlap risk. Twin grouping may miss weaker similarity, so these are upper bounds, not evidence for new patients.
- Done once. No model, feature or setting was changed after seeing it.